# 01 - 配对筛选 / Pair Screening

本 Notebook 展示如何用 `strategy.screen_pairs` 在 A 股样本宇宙中挑出协整对。
运行前请先执行 `python scripts/init_db.py --use-samples` 灌入样本数据。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from sqlalchemy import text
from core.data import IngestService
from strategy import screen_pairs

svc = IngestService()
prices = svc.load_ohlcv()
with svc.engine.begin() as conn:
    stocks = pd.read_sql(text('SELECT * FROM stocks'), conn)
print('loaded', len(prices), 'rows /', len(stocks), 'stocks')
stocks.head()

In [ ]:
pairs = screen_pairs(prices, stocks, pvalue_threshold=0.10, max_pairs=20)
print(f'found {len(pairs)} cointegrated pairs')
from strategy.pair_selection import to_dataframe
to_dataframe(pairs).head()

In [ ]:
import matplotlib.pyplot as plt
pair = pairs[0]
wide = prices.pivot(index='trade_date', columns='ts_code', values='close').sort_index()
ax = wide[[pair.code_a, pair.code_b]].plot(figsize=(10, 4), title=f'{pair.code_a} vs {pair.code_b}')
ax.set_ylabel('price'); plt.tight_layout()